In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [15]:
import peft

In [5]:
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-1b4-zh", low_cpu_mem_usage = True)
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-1b4-zh")

my-lora is an adapter pretrained by the instructor.

In [6]:
p_model = PeftModel.from_pretrained(model, model_id = "./my-lora/")

`get_peft_model(model, config)` is used to initialize a PEFT model for training.

This function takes:
- A base pretrained model (e.g. LlamaForCausalLM)
- A PEFT configuration (e.g. LoRA, Prompt Tuning, etc.)
- and wraps the base model to make only specific parameters trainable (based on the config).

`PeftModel.from_pretrained(model, "./my-lora/")` is used to load an already-trained PEFT adapter into a base model.

This function:
- Takes a base model (same architecture as used for training)
- Loads the LoRA/Prompt/P-Tuning weights that were previously saved in the adapter directory

In [7]:
type(p_model)

peft.peft_model.PeftModelForCausalLM

In [17]:
peft.peft_model.PeftModelForCausalLM??

Init signature:
peft.peft_model.PeftModelForCausalLM(
    model: 'torch.nn.Module',
    peft_config: 'PeftConfig',
    adapter_name: 'str' = 'default',
    **kwargs,
) -> 'None'
Source:        
class PeftModelForCausalLM(PeftModel):
    """
    Peft model for causal language modeling.

    Args:
        model ([`~transformers.PreTrainedModel`]): Base transformer model.
        peft_config ([`PeftConfig`]): Peft config.
        adapter_name (`str`,  *optional*): The name of the adapter, defaults to `"default"`.
        autocast_adapter_dtype (`bool`, *optional*):
            Whether to autocast the adapter dtype. Defaults to `True`. Right now, this will only cast adapter weights
            using float16 and bfloat16 to float32, as this is typically required for stable training, and only affect
            select PEFT tuners.

    Example:

        ```py
        >>> from transformers import AutoModelForCausalLM
        >>> from peft import PeftModelForCausalLM, get_peft_config

        

`peft.peft_model.PeftModelForCausalLM` is an adapted version of `torch.nn.Module`, specifically designed to wrap a causal language model (like GPT-style models) with parameter-efficient fine-tuning (PEFT) capabilities.

In [8]:
p_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(46145, 2048)
        (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0-23): 24 x BloomBlock(
            (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=6144, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=6144, bias=False)
                )
                (lora_embedding_A): Paramete

In [10]:
p_model.device

device(type='cpu')

In [12]:
p_model = p_model.to("mps")
p_model.device

device(type='mps', index=0)

## Use the model for reasonaing.

In [26]:
ipt = tokenizer("Human: {}\n{}".format("如何备战高考", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
# 把model输出的response结果再次转为文本
print(tokenizer.decode(p_model.generate(**ipt, max_length=256, do_sample=True)[0], skip_special_tokens=True))

Human: 如何备战高考

Assistant: 高考报名
高考报名指的是高考报名。考试报名指的是考试报名。
高考报名指的是高考报名。报名时间：考试报名指的是考试报名。报名时间：高考每年两次，分别为5月中的第一次报名时间，7月中的第二次考试报名时间。
高考报名指的是高考报名。报名时间：考试报名指的是考试报名。报名时间：2020年的5月26日，在6月6日-23日，在6月10日-18日进行第一次参加考试报名。报名时间：2020年的7月4日，在6月10日-18日，在6月10日-18日进行第二次参加考试报名。报名时间：高考报名是每年两次，分别为5月中的第一次报名时间，7月中的第二次考试报名时间。在考试报名时间公布之后，各个高校根据考生的高考成绩进行招生录取。报名时间：2021年的5月18日进行第一次考试报名。报名时间：2021年的7月24日进行第二次考试报名。报名时间：高考报名一般是每年两次，分别为春季和秋季的两次。在6月份和7月下旬，各个高校都会对考生进行入学测试并公布成绩，根据成绩进行录取。报名时间：高考报名一般为每年的两次。春季和秋季报名时间分别为4月和8月，7月末和


In [20]:
ipt

{'input_ids': tensor([[26283,    29, 28683,  7189, 31022,   189,   189,  4340, 17245,    29,
           210]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}

## Merge Model Parameter

In [23]:
merge_model = p_model.merge_and_unload()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [24]:
merge_model

BloomForCausalLM(
  (transformer): BloomModel(
    (word_embeddings): Embedding(46145, 2048)
    (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
    (h): ModuleList(
      (0-23): 24 x BloomBlock(
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (self_attention): BloomAttention(
          (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
          (dense): Linear(in_features=2048, out_features=2048, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (mlp): BloomMLP(
          (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
          (gelu_impl): BloomGelu()
          (dense_4h_to_h): Linear(in_features=8192, out_features=2048, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
  )
  (l

#### The LoRA adapter is merged with original model

In [25]:
merge_model = merge_model.to("mps")

In [27]:
ipt = tokenizer("Human: {}\n{}".format("如何备战高考", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
# 把model输出的response结果再次转为文本
print(tokenizer.decode(merge_model.generate(**ipt, max_length=256, do_sample=True)[0], skip_special_tokens=True))

Human: 如何备战高考

Assistant: 高考的备战可分三个步骤，一是针对性备考，二是专项复习，三是日常作息。

备考针对性：高考备考是每个同学都必须经历和持续做的工作，针对性是高考备考的前提。针对性备考的目的是为高三学子的高考试成绩保驾护航，提高考生备考的成效。

专项复习：高考备考需要对重点科目，难点考点进行逐一归纳梳理，对知识进行系统重组，提升知识吸收力以及熟练程度，从而达到提高备考效率的效果。

日常作息：高考备考需要严格按照高考作息规律安排，保证复习效果。

备考应做哪些准备？

首先，对高三这一阶段的整个知识体系做梳理，重点进行重要难点知识的详细讲解和讲解，这样便于考生高效的完成复习任务；

其次，对于高考复习方法进行全面总结和归纳分类，这样便于考生高效的进行复习；

再次，要做好高考试卷的模拟和习题测试，并且做好详细的自我分析，以便于及时调整方向；

最后，还要养成


## Save merged model

In [28]:
merge_model.save_pretrained("./chatbot/merge_model")

This will save the entire merged model, same size as the original model, not only the adapter. This will roughly occupy 5.2GB.